In [ ]:
import sys

sys.path.append("..")

from datetime import datetime
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from nnspike.constants import RELATIVE_POSITION_SCALE, ROI_CNN
from nnspike.data import MultiTaskDataset, balance_dataset
from nnspike.models import MultiTaskLoss, NvidiaModelMultiTask
from notebooks.utils import view_data_distribution

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_course = "right"
date_label = datetime.today().strftime("%m%d")

print(f"Training Course: {train_course}; Date Label: {date_label}")

## Loading Dataset

In [ ]:
label_paths = glob("../storage/labels/*.csv")
# label_paths = [path for path in label_paths
#                  if os.path.basename(path)[:8] < "20250801"]

df = pd.DataFrame()
for label_path in label_paths:
    
    label_df = pd.read_csv(label_path)
    df = pd.concat([df, label_df])

df = df[df['use']==True]

# Drop the unlabeled rows
df = df.dropna(subset=['target_x', 'mode'])

# Set values in 'mode' to 5 where the value is greater than 5
df.loc[df['mode'] > 5, 'mode'] = 5

df["mode"] = df["mode"].astype(int)

# Ensure invalid data were not existed
extracted_df = df[df['target_x'].isna()]
if extracted_df is not None and not extracted_df.empty:
    raise Exception("Extracted dataframe is not None or not empty!")

print(f"Total number of training records: {len(df)}")
print(f"Unique behavior mode: {df['mode'].unique()}")

## Dataset Distribution

In [ ]:
view_data_distribution(df, ['mode', 'motor_b_relative_position'], [5, 30])

## Dataset Preparation

In [ ]:
image_paths = df['image_path'].to_list()

combined_positions = abs(df['motor_a_relative_position']) + abs(df['motor_b_relative_position'])
relative_positions = (combined_positions/RELATIVE_POSITION_SCALE).to_list()
courses = df['course'].to_list()
target_xs = df['target_x'].to_list()
modes = df['mode'].to_list()

X_all = [[x, y, z] for x, y, z in zip(image_paths, relative_positions, courses)]
y_all = [[x, y] for x, y in zip(modes, target_xs)] 

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=6)

train_set = MultiTaskDataset(inputs=X_train, outputs=y_train, roi=ROI_CNN, train_course=train_course)
val_set = MultiTaskDataset(inputs=X_val, outputs=y_val, roi=ROI_CNN, train_course=train_course)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=32, shuffle=True)

## Loss Function, Optimizer, Model Initialization

In [ ]:
model_label = "nvidia"

criterion = MultiTaskLoss(mode_weight=1.0, control_weight=30.0,control_scale=10.0)
model = NvidiaModelMultiTask(num_modes=6)
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# Initialize TensorBoard writer
writer = SummaryWriter()

# Training loop
num_epochs = 20
best_val_loss = float('inf')
best_model_state = None
best_epoch = 0

# Training loop
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_mode_loss = 0.0
    train_control_loss = 0.0
    
    for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        optimizer.zero_grad()
        inputs = [input_tensor.to(device) for input_tensor in inputs]
        
        # Assuming labels is a tuple/list: (mode_labels, control_labels)
        mode_labels = labels[0].to(device)  # Shape: (batch_size,) with class indices
        control_labels = labels[1].to(device)  # Shape: (batch_size, 1) or (batch_size,)
        
        outputs = model(inputs[0], inputs[1])
        
        # Calculate multi-task loss
        loss, mode_loss, control_loss = criterion(outputs, (mode_labels, control_labels))
    
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_mode_loss += mode_loss.item()
        train_control_loss += control_loss.item()
    
    # Calculate average losses
    avg_train_loss = train_loss / len(train_loader)
    avg_train_mode_loss = train_mode_loss / len(train_loader)
    avg_train_control_loss = train_control_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_mode_loss = 0.0
    val_control_loss = 0.0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = [input_tensor.to(device) for input_tensor in inputs]
            mode_labels = labels[0].to(device)
            control_labels = labels[1].to(device)
            
            outputs = model(inputs[0], inputs[1])
            loss, mode_loss, control_loss = criterion(outputs, (mode_labels, control_labels))
            
            val_loss += loss.item()
            val_mode_loss += mode_loss.item()
            val_control_loss += control_loss.item()

    # Calculate average validation losses
    avg_val_loss = val_loss / len(val_loader)
    avg_val_mode_loss = val_mode_loss / len(val_loader)
    avg_val_control_loss = val_control_loss / len(val_loader)

    # Check if this is the best model so far
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict().copy()  # Deep copy of model state
        best_epoch = epoch + 1
        torch.save(model.state_dict(), f"../storage/models/{model_label}_multi_{train_course}_{date_label}_{best_epoch}.pt")
        print(f'  New best model found at epoch {best_epoch}!')
    
    # Log to TensorBoard
    writer.add_scalar('Loss/train_total', avg_train_loss, epoch)
    writer.add_scalar('Loss/train_mode', avg_train_mode_loss, epoch)
    writer.add_scalar('Loss/train_control', avg_train_control_loss, epoch)
    writer.add_scalar('Loss/val_total', avg_val_loss, epoch)
    writer.add_scalar('Loss/val_mode', avg_val_mode_loss, epoch)
    writer.add_scalar('Loss/val_control', avg_val_control_loss, epoch)
    
    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'  Train - Total: {avg_train_loss:.5f}, Mode: {avg_train_mode_loss:.5f}, Control: {avg_train_control_loss:.5f}')
    print(f'  Val   - Total: {avg_val_loss:.5f}, Mode: {avg_val_mode_loss:.5f}, Control: {avg_val_control_loss:.5f}')
# Close TensorBoard writer
writer.close()

print("\nTraining completed! Feature maps have been logged to TensorBoard.")
print("To view the visualizations, run: tensorboard --logdir=runs")

## Export to ONNX Model

In [ ]:
onnx_path = f"../storage/models/{model_label}_multi_{train_course}_{date_label}.onnx"

model.eval()

# Create dummy inputs - adjust dimensions as needed
dummy_image = torch.randn(1, 3, 66, 200)
dummy_relative_position = torch.randn(1, 1)

# Export to ONNX
torch.onnx.export(
    model,
    (dummy_image, dummy_relative_position),
    onnx_path,
    export_params=True,
    opset_version=11,
    input_names=["image", "relative_position"],
    output_names=["mode_output", "control_output"],
)

## Example Usage

In [ ]:
import onnxruntime as ort
import numpy as np

# Load the ONNX model
onnx_path = f"../storage/models/{model_label}_multi_{train_course}_{date_label}.onnx"
session = ort.InferenceSession(onnx_path)

# Create dummy inputs (same as during export)
dummy_image = np.random.randn(1, 3, 66, 200).astype(np.float32)
dummy_relative_position = np.random.randn(1, 1).astype(np.float32)

# Prepare inputs dictionary
inputs = {
    "image": dummy_image,
    "relative_position": dummy_relative_position
}

# Run inference
outputs = session.run(["mode_output", "control_output"], inputs)

# Get results
# Get results
logits = outputs[0]  # shape: [batch_size, num_classes]
probabilities = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)  # softmax

# Get predicted classes and their confidence scores
predicted_classes = np.argmax(probabilities, axis=1)
confidence_scores = np.max(probabilities, axis=1)

control_output = outputs[1]

print(f"Mode output shape: {mode_output.shape}")
print(f"Mode output: {mode_output}")
print(f"Predicted classes: {predicted_classes}")
print(f"Confidence scores: {confidence_scores}")
print(f"Control output shape: {control_output.shape}")
print(f"Control output: {control_output}")